In [1]:
import subprocess
pkgs = ["faiss-cpu", "sentence-transformers", "pdfplumber",
        "pypdf", "langchain", "langchain-community",
        "langchain-huggingface", "transformers",
        "accelerate", "fpdf2", "tqdm", "gradio"]
for pkg in pkgs:
    r = subprocess.run(["pip", "install", pkg, "-q", "--no-cache-dir"],
                       capture_output=True, text=True)
    print(f"{'✅' if r.returncode == 0 else '❌'} {pkg}")

✅ faiss-cpu
✅ sentence-transformers
✅ pdfplumber
✅ pypdf
✅ langchain
✅ langchain-community
✅ langchain-huggingface
✅ transformers
✅ accelerate
✅ fpdf2
✅ tqdm
✅ gradio


In [2]:
import sys
sys.path.insert(0, "/workspace/shared/audit_validator")

from src.rule_loader import load_all_rules, load_rules_as_text
from src.rag_pipeline import build_faiss_index, save_index

rules      = load_all_rules()
rule_texts = load_rules_as_text(rules)
index, rule_texts_indexed = build_faiss_index(rule_texts)
save_index(index, rule_texts_indexed)

print(f"\n✅ Index rebuilt with {len(rules)} rules across {len(set(r['framework'] for r in rules))} frameworks")
print(f"Frameworks: {set(r['framework'] for r in rules)}")
print(f"Index vectors: {index.ntotal}")

Loaded 22 rules from 5 frameworks
Loading embedding model on: cuda
Embedding model loaded: BAAI/bge-small-en-v1.5


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Embedding rules] 18.19s
FAISS index built: 22 vectors | dim=384
Index saved to: /workspace/shared/audit_validator/data/vector_store

✅ Index rebuilt with 22 rules across 5 frameworks
Frameworks: {'PCI_DSS', 'Insurance_Compliance', 'HIPAA', 'SOX', 'GDPR'}
Index vectors: 22


In [3]:
import sys, importlib
sys.path.insert(0, "/workspace/shared/audit_validator")

# Reload all modules fresh
import src.rule_loader, src.rag_pipeline, src.validator
import src.confidence_scorer, src.report_generator, src.validator_agent
for mod in [src.rule_loader, src.rag_pipeline, src.validator,
            src.confidence_scorer, src.report_generator, src.validator_agent]:
    importlib.reload(mod)

from src.validator_agent import AuditAgent

agent  = AuditAgent()
result = agent.run(
    "/workspace/shared/audit_validator/data/sample_docs/demo_contract.txt",
    max_chunks=6
)

🤖 AuditAgent initializing...
Index loaded: 22 vectors
✅ AuditAgent ready.

📄 Starting audit: /workspace/shared/audit_validator/data/sample_docs/demo_contract.txt
[Document parse] 0.001s
   Words: 267 | Chunks: 1
   (Limited to 6 chunks for speed)

Validating chunk 1/1...
Loading Qwen/Qwen2.5-7B-Instruct ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


[GPU Memory] After LLM load
  Allocated : 15.45 GB
  Reserved  : 15.72 GB
  Total     : 206.14 GB
  Free      : 190.42 GB
LLM ready: Qwen/Qwen2.5-7B-Instruct
[LLM inference] 20.704s
[Full RAG + LLM validation] 27.727s
[Confidence scoring] 0.0s
✅ HTML report saved: /workspace/shared/audit_validator/outputs/audit_reports/audit_report_20260610_062425.html
✅ JSON report saved: /workspace/shared/audit_validator/outputs/audit_reports/audit_report_20260610_062425.json
[Report generation] 0.008s

✅ AUDIT COMPLETE
  Overall Score  : 0.0%
  Risk Level     : CRITICAL
  Rules Checked  : 3
  Total Tokens   : 843
  Avg Latency    : 27.71s/chunk

  Framework Scores:
    GDPR: 0.0%
    SOX: 0.0%

  Status Breakdown:
    NON_COMPLIANT: 3

  📁 HTML Report: /workspace/shared/audit_validator/outputs/audit_reports/audit_report_20260610_062425.html


In [4]:
import json
print("Frameworks hit:", result["score_data"]["framework_scores"])
print("\nSample reasoning:", result["validations"][0].get("reasoning", "MISSING"))

Frameworks hit: {'GDPR': 0.0, 'SOX': 0.0}

Sample reasoning: The document states data is retained indefinitely without a deletion schedule.


In [5]:
# Check what statuses actually came back
from collections import Counter
statuses = Counter(v.get("status") for v in result["validations"])
print("Status counts:", statuses)
print("\nAll validations:")
for v in result["validations"]:
    print(f"  {v.get('rule_id')} | {v.get('status')} | confidence: {v.get('confidence_score')} | reasoning: {v.get('reasoning','')[:80]}")

Status counts: Counter({'NON_COMPLIANT': 3})

All validations:
  GDPR-002 | NON_COMPLIANT | confidence: 85 | reasoning: The document states data is retained indefinitely without a deletion schedule.
  GDPR-005 | NON_COMPLIANT | confidence: 90 | reasoning: Data processing agreements have not been established with all third-party vendor
  SOX-003 | NON_COMPLIANT | confidence: 80 | reasoning: Financial records are only retained for 3 years, not the required 7 years.


In [6]:
import sys, importlib
sys.path.insert(0, "/workspace/shared/audit_validator")

import src.rule_loader, src.rag_pipeline, src.validator
import src.confidence_scorer, src.report_generator, src.validator_agent
for mod in [src.rule_loader, src.rag_pipeline, src.validator,
            src.confidence_scorer, src.report_generator, src.validator_agent]:
    importlib.reload(mod)

from src.validator_agent import AuditAgent

agent  = AuditAgent()
result = agent.run(
    "/workspace/shared/audit_validator/data/sample_docs/demo_contract_v2.txt",
    max_chunks=8
)

🤖 AuditAgent initializing...
Index loaded: 22 vectors
✅ AuditAgent ready.

📄 Starting audit: /workspace/shared/audit_validator/data/sample_docs/demo_contract_v2.txt
[Document parse] 0.002s
   Words: 458 | Chunks: 2
   (Limited to 8 chunks for speed)

Validating chunk 1/2...
Loading Qwen/Qwen2.5-7B-Instruct ...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


[GPU Memory] After LLM load
  Allocated : 15.58 GB
  Reserved  : 15.95 GB
  Total     : 206.14 GB
  Free      : 190.19 GB
LLM ready: Qwen/Qwen2.5-7B-Instruct
[LLM inference] 8.454s

Validating chunk 2/2...
[LLM inference] 7.458s
[Full RAG + LLM validation] 22.754s
[Confidence scoring] 0.0s
✅ HTML report saved: /workspace/shared/audit_validator/outputs/audit_reports/audit_report_20260610_063341.html
✅ JSON report saved: /workspace/shared/audit_validator/outputs/audit_reports/audit_report_20260610_063341.json
[Report generation] 0.005s

✅ AUDIT COMPLETE
  Overall Score  : 50.0%
  Risk Level     : HIGH
  Rules Checked  : 6
  Total Tokens   : 1617
  Avg Latency    : 11.36s/chunk

  Framework Scores:
    GDPR: 50.0%
    PCI_DSS: 75.0%
    HIPAA: 0.0%

  Status Breakdown:
    COMPLIANT: 2
    NON_COMPLIANT: 2
    PARTIAL: 2

  📁 HTML Report: /workspace/shared/audit_validator/outputs/audit_reports/audit_report_20260610_063341.html


In [7]:
from collections import Counter
statuses = Counter(v.get("status") for v in result["validations"])

print("=== AUDIT RESULTS ===")
print(f"Overall Score  : {result['score_data']['overall_score']}%")
print(f"Risk Level     : {result['score_data']['risk_level']}")
print(f"Framework Scores: {result['score_data']['framework_scores']}")
print(f"\nStatus Counts  : {dict(statuses)}")
print(f"\nSample reasoning: {result['validations'][0].get('reasoning','')}")


=== AUDIT RESULTS ===
Overall Score  : 50.0%
Risk Level     : HIGH
Framework Scores: {'GDPR': 50.0, 'PCI_DSS': 75.0, 'HIPAA': 0.0}

Status Counts  : {'COMPLIANT': 2, 'NON_COMPLIANT': 2, 'PARTIAL': 2}

Sample reasoning: The document clearly states the lawful basis for processing.


In [8]:
from datetime import datetime
import json

day5_log = {
    "day":           5,
    "date":          datetime.now().isoformat(),
    "improvements":  [
        "Added HIPAA rules (5 rules)",
        "Added PCI-DSS rules (4 rules)",
        "Total rules: 22 across 5 frameworks",
        "Added reasoning field to LLM output",
        "Risk-ranked remediation action plan in HTML report",
        "Visual score bars in HTML report",
        "Reasoning column added to Gradio table"
    ],
    "total_rules":   22,
    "frameworks":    ["GDPR", "SOX", "HIPAA", "PCI_DSS", "Insurance_Compliance"],
    "status":        "complete"
}

with open("/workspace/shared/audit_validator/logs/day5_summary.json", "w") as f:
    json.dump(day5_log, f, indent=2)

print(json.dumps(day5_log, indent=2))
print("\n✅ Day 5 complete!")

{
  "day": 5,
  "date": "2026-06-10T06:35:31.120311",
  "improvements": [
    "Added HIPAA rules (5 rules)",
    "Added PCI-DSS rules (4 rules)",
    "Total rules: 22 across 5 frameworks",
    "Added reasoning field to LLM output",
    "Risk-ranked remediation action plan in HTML report",
    "Visual score bars in HTML report",
    "Reasoning column added to Gradio table"
  ],
  "total_rules": 22,
  "frameworks": [
    "GDPR",
    "SOX",
    "HIPAA",
    "PCI_DSS",
    "Insurance_Compliance"
  ],
  "status": "complete"
}

✅ Day 5 complete!
